# Quantalytics - Gold Layer Pipeline
Transforms Silver tables into star/snowflake dimensional models and operational tables optimized for BI queries.

In [ ]:
import json
import uuid
from datetime import date, datetime, timedelta, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import LongType, StringType, StructField, StructType, TimestampType

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "workspace"
IBM_SILVER = f"{CATALOG}.silver.ibm_qubit_calibration"
NOAA_SILVER = f"{CATALOG}.silver.noaa_environment"
LOG_TABLE = f"{CATALOG}.silver.pipeline_execution_logs"
BAD_TABLE = f"{CATALOG}.silver.quarantined_records"
GOLD = f"{CATALOG}.gold"

def make_widget(name, kind, default, choices=None):
    try:
        dbutils.widgets.get(name)
    except Exception:
        if kind == "dropdown":
            dbutils.widgets.dropdown(name, default, choices)
        else:
            dbutils.widgets.text(name, default)

make_widget("mode", "dropdown", "full", ["full", "incremental"])
make_widget("start_date", "text", "")
make_widget("end_date", "text", "")
make_widget("run_id", "text", "")

MODE = dbutils.widgets.get("mode").lower().strip()
START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()
RUN_ID = dbutils.widgets.get("run_id").strip() or str(uuid.uuid4())

if MODE not in ("full", "incremental"):
    raise ValueError("mode must be full or incremental")

for value in (START_DATE, END_DATE):
    if value:
        datetime.strptime(value, "%Y-%m-%d")

for table in (IBM_SILVER, NOAA_SILVER, LOG_TABLE, BAD_TABLE):
    if not spark.catalog.tableExists(table):
        raise ValueError(f"Missing prerequisite table: {table}")

LAST_SUCCESS = spark.sql(f"""
    SELECT MAX(finished_at) AS finished_at
    FROM {LOG_TABLE}
    WHERE stage = 'Silver-to-Gold'
      AND dataset = 'gold'
      AND status = 'SUCCESS'
""").first()["finished_at"]

if MODE == "incremental" and not START_DATE:
    if LAST_SUCCESS:
        START_DATE = (LAST_SUCCESS.date() - timedelta(days=2)).isoformat()
    else:
        MODE = "full"

if MODE == "incremental" and not END_DATE:
    END_DATE = date.today().isoformat()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")
RUN_START = datetime.now(timezone.utc).replace(tzinfo=None)

print({
    "run_id": RUN_ID,
    "mode": MODE,
    "start_date": START_DATE or "ALL",
    "end_date": END_DATE or "ALL",
    "last_success": str(LAST_SUCCESS) if LAST_SUCCESS else None,
})

In [ ]:
LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("stage", StringType(), False),
    StructField("dataset", StringType(), False),
    StructField("mode", StringType(), False),
    StructField("parameters", StringType(), True),
    StructField("started_at", TimestampType(), False),
    StructField("finished_at", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("input_rows", LongType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_message", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

spark.createDataFrame([], LOG_SCHEMA).write.format("delta").mode("ignore").saveAsTable(LOG_TABLE)

def audit(status, started, input_rows=0, rows_written=0, details=None, error=None):
    finished = datetime.now(timezone.utc).replace(tzinfo=None)
    parameters = json.dumps(
        {
            "start_date": START_DATE or "ALL",
            "end_date": END_DATE or "ALL",
            **(details or {}),
        },
        sort_keys=True,
    )
    row = [((
        RUN_ID,
        "Silver-to-Gold",
        "gold",
        MODE,
        parameters,
        started,
        finished,
        status,
        int(input_rows),
        int(rows_written),
        0,
        0,
        (error[:1500] if error else None),
        finished,
    ))]
    spark.createDataFrame(row, LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(LOG_TABLE)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.vw_noaa_kp_canonical AS
SELECT
    record_key,
    source,
    observation_type,
    observation_timestamp,
    kp_index,
    solar_flux,
    source_unit,
    source_file,
    load_timestamp
FROM {NOAA_SILVER}
WHERE observation_type = 'planetary_kp'
  AND source_file LIKE 'noaa_full_%'
  AND observation_timestamp IS NOT NULL
""")

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD}.vw_noaa_f107_daily AS
WITH ranked AS (
    SELECT
        record_key,
        source,
        observation_type,
        observation_timestamp,
        kp_index,
        solar_flux,
        source_unit,
        source_file,
        load_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY to_date(observation_timestamp)
            ORDER BY
                CASE WHEN source_file LIKE 'noaa_full_%' THEN 0 ELSE 1 END,
                observation_timestamp DESC
        ) AS rn
    FROM {NOAA_SILVER}
    WHERE observation_type = 'f10_7_flux'
      AND observation_timestamp IS NOT NULL
)
SELECT
    record_key,
    source,
    observation_type,
    to_timestamp(to_date(observation_timestamp)) AS observation_timestamp,
    kp_index,
    solar_flux,
    source_unit,
    source_file,
    load_timestamp
FROM ranked
WHERE rn = 1
""")

In [ ]:
def merge_dim(df, table, key):
    if not spark.catalog.tableExists(table):
        df.limit(0).write.format("delta").saveAsTable(table)
    if df.limit(1).count() == 0:
        return 0
    assignment = {column: f"s.`{column}`" for column in df.columns}
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), f"t.`{key}` = s.`{key}`")
        .whenMatchedUpdate(set=assignment)
        .whenNotMatchedInsertAll()
        .execute())
    return df.count()

bounds = spark.sql(f"""
    SELECT MIN(d) AS min_d, MAX(d) AS max_d
    FROM (
        SELECT to_date(calibration_timestamp) AS d FROM {IBM_SILVER}
        UNION ALL
        SELECT to_date(observation_timestamp) AS d FROM {NOAA_SILVER}
        UNION ALL
        SELECT to_date(started_at) AS d FROM {LOG_TABLE}
        UNION ALL
        SELECT to_date(load_timestamp) AS d FROM {BAD_TABLE}
    )
    WHERE d IS NOT NULL
""").first()

min_d = bounds["min_d"] or date(2024, 1, 1)
max_d = bounds["max_d"] or date.today()

dim_date = spark.sql(f"""
    SELECT
        CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
        d AS full_date,
        year(d) AS year,
        quarter(d) AS quarter,
        month(d) AS month,
        date_format(d, 'MMMM') AS month_name,
        date_format(d, 'yyyy-MM') AS year_month,
        CAST(year(d) * 100 + month(d) AS INT) AS month_sort,
        day(d) AS day_of_month,
        date_format(d, 'EEEE') AS day_name,
        weekofyear(d) AS week_of_year,
        dayofweek(d) IN (1, 7) AS is_weekend
    FROM (
        SELECT explode(sequence(to_date('{min_d}'), to_date('{max_d}'), interval 1 day)) AS d
    )
""")

dim_time = spark.sql("""
    SELECT
        CAST(date_format(ts, 'HHmmss') AS INT) AS time_key,
        hour(ts) AS hour_24,
        minute(ts) AS minute,
        second(ts) AS second,
        date_format(ts, 'HH:mm:ss') AS time_label,
        CAST((hour(ts) * 60 + minute(ts)) DIV 180 AS INT) AS time_bucket_3h,
        hour(ts) AS hour_bucket
    FROM (
        SELECT explode(sequence(
            to_timestamp('1970-01-01 00:00:00'),
            to_timestamp('1970-01-01 23:59:59'),
            interval 1 second
        )) AS ts
    )
""")

dim_backend = (
    spark.table(IBM_SILVER)
    .groupBy("backend_id")
    .agg(
        F.min("calibration_timestamp").alias("first_seen"),
        F.max("calibration_timestamp").alias("last_seen"),
    )
    .withColumn("backend_key", F.sha2(F.col("backend_id"), 256))
    .withColumn("backend_name", F.col("backend_id"))
    .withColumn("provider", F.lit("IBM Quantum"))
    .select("backend_key", "backend_id", "backend_name", "provider", "first_seen", "last_seen")
)

dim_qubit = (
    spark.table(IBM_SILVER)
    .groupBy("backend_id", "qubit_id")
    .agg(
        F.min("calibration_timestamp").alias("first_seen"),
        F.max("calibration_timestamp").alias("last_seen"),
    )
    .withColumn("qubit_key", F.sha2(F.concat_ws("||", "backend_id", "qubit_id"), 256))
    .withColumn("backend_key", F.sha2(F.col("backend_id"), 256))
    .select("qubit_key", "backend_key", "qubit_id", "first_seen", "last_seen")
)

dim_observation_type = (
    spark.createDataFrame(
        [
            ("planetary_kp", "Planetary K-index", "index"),
            ("f10_7_flux", "F10.7 solar radio flux", "sfu"),
        ],
        ["observation_type", "description", "default_unit"],
    )
    .withColumn("observation_type_key", F.sha2("observation_type", 256))
    .select("observation_type_key", "observation_type", "description", "default_unit")
)

dim_stage = (
    spark.sql(f"""
        SELECT DISTINCT stage FROM {LOG_TABLE} WHERE stage IS NOT NULL
        UNION SELECT 'Raw-to-Bronze'
        UNION SELECT 'Bronze-to-Silver'
        UNION SELECT 'Silver-to-Gold'
    """)
    .withColumn("stage_key", F.sha2("stage", 256))
    .select("stage_key", "stage")
)

dim_dataset = (
    spark.sql(f"""
        SELECT DISTINCT dataset FROM {LOG_TABLE} WHERE dataset IS NOT NULL
        UNION SELECT 'ibm'
        UNION SELECT 'noaa'
        UNION SELECT 'gold'
    """)
    .withColumn("dataset_key", F.sha2("dataset", 256))
    .select("dataset_key", "dataset")
)

dim_mode = (
    spark.sql(f"""
        SELECT DISTINCT mode FROM {LOG_TABLE} WHERE mode IS NOT NULL
        UNION SELECT 'full'
        UNION SELECT 'incremental'
        UNION SELECT 'backfill'
    """)
    .withColumn("mode_key", F.sha2("mode", 256))
    .select("mode_key", "mode")
)

dim_status = (
    spark.sql(f"""
        SELECT DISTINCT status FROM {LOG_TABLE} WHERE status IS NOT NULL
        UNION SELECT 'SUCCESS'
        UNION SELECT 'FAILURE'
    """)
    .withColumn("status_key", F.sha2("status", 256))
    .withColumn("is_success", F.col("status") == "SUCCESS")
    .withColumn("is_failure", F.col("status") == "FAILURE")
    .select("status_key", "status", "is_success", "is_failure")
)

dim_error_reason = (
    spark.table(BAD_TABLE)
    .where(F.col("error_reason").isNotNull())
    .select("error_reason")
    .distinct()
    .withColumn("error_reason_key", F.sha2("error_reason", 256))
    .select("error_reason_key", "error_reason")
)

merge_dim(dim_date, f"{GOLD}.dim_date", "date_key")
merge_dim(dim_time, f"{GOLD}.dim_time", "time_key")
merge_dim(dim_backend, f"{GOLD}.dim_backend", "backend_key")
merge_dim(dim_qubit, f"{GOLD}.dim_qubit", "qubit_key")
merge_dim(dim_observation_type, f"{GOLD}.dim_observation_type", "observation_type_key")
merge_dim(dim_stage, f"{GOLD}.dim_stage", "stage_key")
merge_dim(dim_dataset, f"{GOLD}.dim_dataset", "dataset_key")
merge_dim(dim_mode, f"{GOLD}.dim_mode", "mode_key")
merge_dim(dim_status, f"{GOLD}.dim_status", "status_key")
merge_dim(dim_error_reason, f"{GOLD}.dim_error_reason", "error_reason_key")

In [ ]:
def source_window(df, time_column):
    if MODE != "incremental":
        return df
    return df.where(
        F.to_date(F.col(time_column)).between(
            F.to_date(F.lit(START_DATE)),
            F.to_date(F.lit(END_DATE)),
        )
    )

def write_fact(df, table, key):
    row_count = df.count()
    if MODE == "full":
        (df.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table))
    else:
        if not spark.catalog.tableExists(table):
            df.limit(0).write.format("delta").saveAsTable(table)
        assignment = {column: f"s.`{column}`" for column in df.columns}
        (DeltaTable.forName(spark, table).alias("t")
            .merge(df.alias("s"), f"t.`{key}` = s.`{key}`")
            .whenMatchedUpdate(set=assignment)
            .whenNotMatchedInsertAll()
            .execute())
    return row_count

In [ ]:
ibm_source = source_window(spark.table(IBM_SILVER), "calibration_timestamp")
noaa_source = source_window(spark.table(NOAA_SILVER), "observation_timestamp")
logs_source = source_window(spark.table(LOG_TABLE), "started_at")
bad_source = source_window(spark.table(BAD_TABLE), "load_timestamp")

ibm_input = ibm_source.count()
noaa_input = noaa_source.count()
logs_input = logs_source.count()
bad_input = bad_source.count()

fact_qubit = (
    ibm_source
    .withColumn("calibration_key", F.col("record_key"))
    .withColumn("date_key", F.date_format("calibration_timestamp", "yyyyMMdd").cast("int"))
    .withColumn(
        "time_key",
        (
            F.hour("calibration_timestamp") * 10000
            + F.minute("calibration_timestamp") * 100
            + F.second("calibration_timestamp")
        ).cast("int"),
    )
    .withColumn("backend_key", F.sha2("backend_id", 256))
    .withColumn("qubit_key", F.sha2(F.concat_ws("||", "backend_id", "qubit_id"), 256))
    .select(
        "calibration_key",
        "date_key",
        "time_key",
        "backend_key",
        "qubit_key",
        "t1_us",
        "t2_us",
        "readout_error",
        "init_error",
        "operational",
        "source_file",
        F.col("load_timestamp").alias("silver_load_timestamp"),
    )
)

def environment_projection(df):
    return df.select(
        F.col("record_key").alias("environment_key"),
        F.date_format("observation_timestamp", "yyyyMMdd").cast("int").alias("date_key"),
        (
            F.hour("observation_timestamp") * 10000
            + F.minute("observation_timestamp") * 100
            + F.second("observation_timestamp")
        ).cast("int").alias("time_key"),
        F.sha2("observation_type", 256).alias("observation_type_key"),
        "observation_type",
        "kp_index",
        "solar_flux",
        "source",
        "source_file",
        F.col("load_timestamp").alias("silver_load_timestamp"),
    )

fact_environment = (
    environment_projection(spark.table(f"{GOLD}.vw_noaa_kp_canonical"))
    .unionByName(environment_projection(spark.table(f"{GOLD}.vw_noaa_f107_daily")))
)

fact_backend_daily = (
    ibm_source
    .groupBy(F.to_date("calibration_timestamp").alias("full_date"), "backend_id")
    .agg(
        F.countDistinct("calibration_timestamp").alias("calibration_count"),
        F.countDistinct("qubit_id").alias("qubit_count"),
        F.avg("t1_us").alias("avg_t1_us"),
        F.min("t1_us").alias("min_t1_us"),
        F.max("t1_us").alias("max_t1_us"),
        F.avg("t2_us").alias("avg_t2_us"),
        F.min("t2_us").alias("min_t2_us"),
        F.max("t2_us").alias("max_t2_us"),
        F.avg("readout_error").alias("avg_readout_error"),
        F.avg("init_error").alias("avg_init_error"),
        F.avg(F.when(F.col("operational") == True, 1.0).otherwise(0.0)).alias(
            "operational_measurement_pct"
        ),
    )
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("backend_key", F.sha2("backend_id", 256))
    .withColumn(
        "backend_daily_key",
        F.sha2(F.concat_ws("||", "backend_id", F.col("date_key")), 256),
    )
)

environment_table = spark.table(f"{GOLD}.fact_environment")
kp_daily = (
    environment_table
    .where(F.col("observation_type") == "planetary_kp")
    .groupBy("date_key")
    .agg(F.avg("kp_index").alias("avg_kp"), F.max("kp_index").alias("max_kp"))
)
f107_daily = (
    environment_table
    .where(F.col("observation_type") == "f10_7_flux")
    .groupBy("date_key")
    .agg(F.avg("solar_flux").alias("avg_f107_solar_flux"))
)

fact_backend_daily = (
    fact_backend_daily
    .join(kp_daily, "date_key", "left")
    .join(f107_daily, "date_key", "left")
    .select(
        "backend_daily_key",
        "date_key",
        "backend_key",
        "calibration_count",
        "qubit_count",
        "avg_t1_us",
        "min_t1_us",
        "max_t1_us",
        "avg_t2_us",
        "min_t2_us",
        "max_t2_us",
        "avg_readout_error",
        "avg_init_error",
        "operational_measurement_pct",
        "avg_kp",
        "max_kp",
        "avg_f107_solar_flux",
    )
)

fact_pipeline_run = (
    logs_source
    .withColumn("date_key", F.date_format("started_at", "yyyyMMdd").cast("int"))
    .withColumn(
        "time_key",
        (
            F.hour("started_at") * 10000
            + F.minute("started_at") * 100
            + F.second("started_at")
        ).cast("int"),
    )
    .withColumn("stage_key", F.sha2("stage", 256))
    .withColumn("dataset_key", F.sha2("dataset", 256))
    .withColumn("mode_key", F.sha2("mode", 256))
    .withColumn("status_key", F.sha2("status", 256))
    .withColumn(
        "duration_seconds",
        F.greatest(
            F.lit(0),
            F.unix_timestamp("finished_at") - F.unix_timestamp("started_at"),
        ),
    )
    .withColumn(
        "pipeline_run_key",
        F.sha2(
            F.concat_ws("||", "run_id", "stage", "dataset", F.col("started_at").cast("string")),
            256,
        ),
    )
    .select(
        "pipeline_run_key",
        "run_id",
        "date_key",
        "time_key",
        "stage_key",
        "dataset_key",
        "mode_key",
        "status_key",
        "started_at",
        "finished_at",
        "duration_seconds",
        "input_rows",
        "rows_inserted",
        "rows_updated",
        "rows_quarantined",
        "error_message",
        "parameters",
    )
)

fact_quarantine = (
    bad_source
    .withColumn("quarantine_key", F.col("record_key"))
    .withColumn("date_key", F.date_format("load_timestamp", "yyyyMMdd").cast("int"))
    .withColumn(
        "time_key",
        (
            F.hour("load_timestamp") * 10000
            + F.minute("load_timestamp") * 100
            + F.second("load_timestamp")
        ).cast("int"),
    )
    .withColumn("dataset_key", F.sha2("dataset", 256))
    .withColumn("error_reason_key", F.sha2("error_reason", 256))
    .select(
        "quarantine_key",
        "date_key",
        "time_key",
        "dataset_key",
        "error_reason_key",
        "source_file",
        F.col("load_timestamp").alias("quarantined_at"),
    )
)

FACT_TABLES = {
    "fact_qubit_calibration": (
        fact_qubit,
        f"{GOLD}.fact_qubit_calibration",
        "calibration_key",
    ),
    "fact_environment": (
        fact_environment,
        f"{GOLD}.fact_environment",
        "environment_key",
    ),
    "fact_backend_daily": (
        fact_backend_daily,
        f"{GOLD}.fact_backend_daily",
        "backend_daily_key",
    ),
    "fact_pipeline_run": (
        fact_pipeline_run,
        f"{GOLD}.fact_pipeline_run",
        "pipeline_run_key",
    ),
    "fact_quarantine": (
        fact_quarantine,
        f"{GOLD}.fact_quarantine",
        "quarantine_key",
    ),
}

print("Fact dataframes ready.")

In [ ]:
optimizations = {
    f"{GOLD}.fact_qubit_calibration": "date_key, backend_key",
    f"{GOLD}.fact_environment": "date_key, observation_type_key",
    f"{GOLD}.fact_backend_daily": "date_key, backend_key",
    f"{GOLD}.fact_pipeline_run": "date_key, status_key",
    f"{GOLD}.fact_quarantine": "date_key, dataset_key",
}

try:
    FACT_ROWS = {
        name: write_fact(df, table, key)
        for name, (df, table, key) in FACT_TABLES.items()
    }

    for table, columns in optimizations.items():
        spark.sql(f"OPTIMIZE {table} ZORDER BY ({columns})")

    details = {
        "rows_aggregated": int(ibm_input),
        "noaa_rows_aggregated": int(noaa_input),
        "fact_rows": {key: int(value) for key, value in FACT_ROWS.items()},
    }
    audit(
        "SUCCESS",
        RUN_START,
        input_rows=ibm_input + noaa_input + logs_input + bad_input,
        rows_written=sum(FACT_ROWS.values()),
        details=details,
    )
    print("Gold build succeeded.", FACT_ROWS)
except Exception as exc:
    audit("FAILURE", RUN_START, error=str(exc))
    raise

In [ ]:
display(spark.sql(f"""
    SELECT 'fact_qubit_calibration' AS table_name, COUNT(*) AS rows FROM {GOLD}.fact_qubit_calibration
    UNION ALL
    SELECT 'fact_environment', COUNT(*) FROM {GOLD}.fact_environment
    UNION ALL
    SELECT 'fact_backend_daily', COUNT(*) FROM {GOLD}.fact_backend_daily
    UNION ALL
    SELECT 'fact_pipeline_run', COUNT(*) FROM {GOLD}.fact_pipeline_run
    UNION ALL
    SELECT 'fact_quarantine', COUNT(*) FROM {GOLD}.fact_quarantine
"""))